### Tools
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute

In [1]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv()

# Create model
model = init_chat_model(
    "groq:llama-3.3-70b-versatile"
)

# Normal LLM call
response = model.invoke("Why do parrots talk?")

print(response.content)

Parrots are known for their remarkable ability to mimic human speech and other sounds, but why do they do it? The answer lies in their evolution, social behavior, and communication needs.

**Evolutionary advantages:**

1. **Mimicry as a survival strategy**: In the wild, parrots use vocalizations to communicate with each other, warn other birds of predators, and attract mates. Mimicking other sounds, like those of other birds or even predators, can help them blend in, avoid danger, or gain an advantage.
2. **Social learning**: Parrots are highly social birds that live in flocks. By mimicking each other's vocalizations, they can learn and adopt new sounds, which helps to strengthen social bonds and reinforce their position within the flock.

**Communication needs:**

1. **Contact calls**: Parrots use vocalizations to maintain contact with their flock members, especially when they're foraging or flying. By mimicking human speech, they may be attempting to initiate interaction or respond t

In [10]:
# -------------------------
# Define tool
# -------------------------

@tool
def get_weather(location: str) -> str:
    """Get the weather for a location."""
    return f"The weather is sunny in {location}."


# Give tool to model
model_with_tools = model.bind_tools([get_weather])

In [11]:
messages = [
    {
        "role": "user",
        "content": "What is the weather in Boston?"
    }
]

# -------------------------
# STEP 1 — Ask model
# -------------------------

ai_msg = model_with_tools.invoke(messages)

print("\nAI MESSAGE:")
print(ai_msg)

# IMPORTANT
messages.append(ai_msg)




AI MESSAGE:
content='' additional_kwargs={'tool_calls': [{'id': 'sh22ct5fg', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 219, 'total_tokens': 233, 'completion_time': 0.042121374, 'completion_tokens_details': None, 'prompt_time': 0.02887659, 'prompt_tokens_details': None, 'queue_time': 0.008524409, 'total_time': 0.070997964}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fe851-6a23-7542-87d9-e40f3cddfac6-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'sh22ct5fg', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 219, 'output_tokens': 14, 'total_tokens': 233}


### Tool Execution Loop

In [13]:
# -------------------------
# STEP 2 — Execute tools
# -------------------------

for tool_call in ai_msg.tool_calls:

    if tool_call["name"] == "get_weather":

        tool_result = get_weather.invoke(
            tool_call["args"]
        )

        print("\nTOOL RESULT:")
        print(tool_result)

        messages.append(tool_result)


# -------------------------
# STEP 3 — Ask model again
# -------------------------

final_response = model_with_tools.invoke(messages)

print("\nFINAL RESPONSE OBJECT:")
print(final_response)

# print("\nFINAL RESPONSE:")
# print(repr(final_response.content))


TOOL RESULT:
The weather is sunny in Boston.

FINAL RESPONSE OBJECT:
content='' additional_kwargs={'tool_calls': [{'id': '8hj404hs4', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 261, 'total_tokens': 275, 'completion_time': 0.017011863, 'completion_tokens_details': None, 'prompt_time': 0.013392224, 'prompt_tokens_details': None, 'queue_time': 0.008357334, 'total_time': 0.030404087}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fe852-5eae-7490-8f25-423463a432fa-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '8hj404hs4', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 261, 'output_tokens': 14, 'total_tokens': 275}


### Basic AI Agent Using Simple Python

In [20]:
def calculator(expression):
    return eval(expression)
    

In [21]:
def weather(city):
    weather_data = {
        "lahore": "32, Sunny",
        "karachi": "31, Humid",
        "islamabad": "28, Cloudy"

    }

    return weather_data.get(
        city.lower(),
        "Weather not available"
    )

In [22]:
from copy import replace
class Agent:

    def think(self, question):

        if "*" in question:
            return "calculator"

        if "weather" in question.lower():
            return "weather"

        return "chat"


    def execute_tool(self, tool, question):

        if tool == "calculator":
            return calculator(question)

        if tool == "weather":
            city = question.lower().replace("weather", "").strip()
            return weather(city)

        return "I don't know."

    def run(self, question):

        tool = self.think(question)

        answer = self.execute_tool(tool, question)

        return answer

    


In [24]:
agent = Agent()
print(agent.run("12*8"))
print(agent.run("Weather Lahore"))

96
32, Sunny
